# MIMIC FHIR Data Explorer

In [1]:
import duckdb
conn = duckdb.connect('mimic_fhir.duckdb', read_only=True)

In [2]:
# Show all tables
conn.sql("SHOW TABLES").show()

┌─────────────────────────┐
│          name           │
│         varchar         │
├─────────────────────────┤
│ condition               │
│ encounter               │
│ location                │
│ medication_dispense     │
│ medication_statement    │
│ observation             │
│ observation_vital_signs │
│ organization            │
│ patient                 │
│ procedure               │
├─────────────────────────┤
│         10 rows         │
└─────────────────────────┘



In [3]:
# View patient schema
conn.sql("DESCRIBE patient").show()

┌──────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │                                                                              column_type                                                                              │  null   │   key   │ default │  extra  │
│       varchar        │                                                                                varchar                                                                                │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ id                   │ UUID                                       

In [4]:
# Sample patients
conn.sql("SELECT * FROM patient LIMIT 5").show()

┌──────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────┬─────────┬────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────

In [5]:
# Patients with encounters
conn.sql("""
SELECT 
    p.name[1].family AS patient_name,
    p.gender,
    p.birthDate,
    e.period.start::TIMESTAMP AS admission,
    e.hospitalization.dischargeDisposition.coding[1].code AS disposition
FROM patient p
JOIN encounter e ON e.subject.reference = 'Patient/' || p.id::VARCHAR
LIMIT 10
""").show()

┌──────────────────┬─────────┬────────────┬─────────────────────┬─────────────┐
│   patient_name   │ gender  │ birthDate  │      admission      │ disposition │
│     varchar      │ varchar │    date    │      timestamp      │   varchar   │
├──────────────────┼─────────┼────────────┼─────────────────────┼─────────────┤
│ Patient_19267214 │ female  │ 2142-12-18 │ 2182-06-11 11:39:00 │ HOME        │
│ Patient_19267214 │ female  │ 2142-12-18 │ 2179-02-20 13:16:00 │ HOME        │
│ Patient_19267214 │ female  │ 2142-12-18 │ 2179-01-16 12:04:00 │ ADMITTED    │
│ Patient_19267214 │ female  │ 2142-12-18 │ 2178-12-18 11:38:00 │ ADMITTED    │
│ Patient_19267214 │ female  │ 2142-12-18 │ 2179-01-20 11:27:00 │ HOME        │
│ Patient_19267214 │ female  │ 2142-12-18 │ 2183-10-08 09:31:00 │ HOME        │
│ Patient_19267214 │ female  │ 2142-12-18 │ 2179-02-16 23:29:00 │ HOME        │
│ Patient_17769122 │ male    │ 2127-11-26 │ 2186-03-22 17:55:00 │ ADMITTED    │
│ Patient_17769122 │ male    │ 2127-11-2

In [6]:
# Convert to pandas DataFrame
df = conn.sql("SELECT * FROM patient LIMIT 100").df()
df.head()

,id,meta,name,gender,birthDate,extension,identifier,resourceType,communication,maritalStatus,deceasedDateTime,managingOrganization
0,0a8eebfd-a352-522e-89f0-1d4a13abdebc,{'profile': ['http://mimic.mit.edu/fhir/mimic/...,"[{'use': 'official', 'family': 'Patient_100000...",female,2128-05-06,[{'url': 'http://hl7.org/fhir/us/core/Structur...,"[{'value': '10000032', 'system': 'http://mimic...",Patient,"[{'language': {'coding': [{'code': 'en', 'syst...","{'coding': [{'code': 'W', 'system': 'http://te...",2180-09-09,{'reference': 'Organization/ee172322-118b-5716...
1,55b28161-ecdb-5c57-ba06-9f0b28e71053,{'profile': ['http://mimic.mit.edu/fhir/mimic/...,"[{'use': 'official', 'family': 'Patient_100000...",male,2088-11-20,[{'url': 'http://hl7.org/fhir/us/core/Structur...,"[{'value': '10000084', 'system': 'http://mimic...",Patient,"[{'language': {'coding': [{'code': 'en', 'syst...","{'coding': [{'code': 'M', 'system': 'http://te...",2161-02-13,{'reference': 'Organization/ee172322-118b-5716...
2,c0ebed23-92f0-56a9-9e05-206ce34dd997,{'profile': ['http://mimic.mit.edu/fhir/mimic/...,"[{'use': 'official', 'family': 'Patient_100001...",male,2138-09-16,[{'url': 'http://hl7.org/fhir/us/core/Structur...,"[{'value': '10000108', 'system': 'http://mimic...",Patient,"[{'language': {'coding': [{'code': 'en', 'syst...","{'coding': [{'code': 'S', 'system': 'http://te...",NaT,{'reference': 'Organization/ee172322-118b-5716...
3,0bdc4fe8-eaa3-5528-ac63-e0aab2933677,{'profile': ['http://mimic.mit.edu/fhir/mimic/...,"[{'use': 'official', 'family': 'Patient_100001...",male,2130-12-10,[{'url': 'http://hl7.org/fhir/us/core/Structur...,"[{'value': '10000115', 'system': 'http://mimic...",Patient,<NA>,"{'coding': [{'code': 'UNK', 'system': 'http://...",NaT,{'reference': 'Organization/ee172322-118b-5716...
4,6c5795f2-6594-52a6-a183-d438112a2f58,{'profile': ['http://mimic.mit.edu/fhir/mimic/...,"[{'use': 'official', 'family': 'Patient_100001...",female,2126-08-14,[{'url': 'http://hl7.org/fhir/us/core/Structur...,"[{'value': '10000117', 'system': 'http://mimic...",Patient,"[{'language': {'coding': [{'code': 'en', 'syst...","{'coding': [{'code': 'D', 'system': 'http://te...",NaT,{'reference': 'Organization/ee172322-118b-5716...
